# Process Simulation and Equipment Design

13 September 2026 revision. The numerical examples use a synthetic teaching basis. Conceptual illustrations represent workflow relationships; they are not measured performance data. The image-generator art is retained with prompts and hashes in `illustrations/imagegen_manifest_2026-09-13.json`; this notebook checks and restores the reviewed raster asset rather than regenerating it through an AI call. Other diagrams and numerical plots are drawn by the maintained illustration code. Set `NEQSIM_PROJECT_ROOT` to the recorded source checkout before running numerical cells.

In [1]:
import os
import sys
from pathlib import Path

start = Path.cwd().resolve()
BOOK_DIR = next(p for p in [start] + list(start.parents)
                if (p / "book.yaml").is_file() and (p / "book_runtime.py").is_file())
sys.path.insert(0, str(BOOK_DIR))
from book_runtime import bootstrap
from build_illustrations import generate_chapter
import json
results = json.loads((BOOK_DIR / "results.json").read_text(encoding="utf-8"))
baseline_record = json.loads((BOOK_DIR / "verification" / "regression_baseline_2026-09-12.json").read_text(encoding="utf-8"))
baselines = baseline_record["outputs"]
assert results["basis"] == baseline_record["basis"], "Review a changed case basis before updating accepted baselines"


In [2]:
jneqsim = bootstrap(os.environ["NEQSIM_PROJECT_ROOT"])
from verify_examples import make_fluid, compression_case, pipeline_case

NeqSim project root: C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim\target\classes
  2. C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\solbraa\OneDrive - NTNU\Documents\GitHub\neqsim\target\neqsim-3.20.0.jar



JVM started: C:\Program Files\Java\graalvm-25.3.4.1+1.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


In [3]:
base, process = compression_case(jneqsim)
assert abs(base["power_kW"] - baselines["compression_base"]["power_kW"]) < 1e-6
print(base)
for expected in baselines["compression_sensitivity"]:
    row, _ = compression_case(jneqsim, pressure=expected["outlet_pressure_bara"])
    for key in ("power_kW", "discharge_temperature_C", "cooler_duty_kW", "cooled_temperature_C"):
        assert abs(row[key] - expected[key]) < 1e-6, (key, row[key], expected[key])
    assert row["mass_balance_relative_error"] < 1e-8
    print(row)


{'outlet_pressure_bara': 120.0, 'polytropic_efficiency': 0.75, 'feed_mass_flow_kg_h': 10000.000000000002, 'gas_mass_flow_kg_h': 10000.000000000002, 'liquid_mass_flow_kg_h': 2e-26, 'power_kW': 337.26908645487555, 'discharge_temperature_C': 93.7492435091761, 'cooled_temperature_C': 35.0, 'cooler_duty_kW': -494.270068598398, 'mass_balance_relative_error': 0.0, 'inlet_enthalpy_rate_kW': -32.5207141541715, 'cooled_outlet_enthalpy_rate_kW': -189.521696297694, 'liquid_outlet_enthalpy_rate_kW': 1.8642372916505865e-28, 'enthalpy_change_kW': -157.0009821435225, 'energy_balance_residual_kW': -5.684341886080802e-14}
{'outlet_pressure_bara': 80.0, 'polytropic_efficiency': 0.75, 'feed_mass_flow_kg_h': 10000.000000000002, 'gas_mass_flow_kg_h': 10000.000000000002, 'liquid_mass_flow_kg_h': 2e-26, 'power_kW': 129.6147986207468, 'discharge_temperature_C': 55.636131748673506, 'cooled_temperature_C': 35.0, 'cooler_duty_kW': -160.35828914395103, 'mass_balance_relative_error': 0.0, 'inlet_enthalpy_rate_kW': 

## Repeat all uncertainty realisations

The low/base/high inputs are teaching assumptions. Each draw performs a complete NeqSim process run. Quantiles use the non-exceedance convention.

In [4]:
import numpy as np
rng = np.random.default_rng(20260912)
powers = []
for _ in range(200):
    flow = float(rng.triangular(9000, 10000, 11000))
    efficiency = float(rng.triangular(0.70, 0.75, 0.80))
    row, _ = compression_case(jneqsim, flow=flow, efficiency=efficiency)
    powers.append(row["power_kW"])
quantiles = np.quantile(powers, [0.1, 0.5, 0.9])
expected = baselines["uncertainty"]["power_kW_quantiles"]
assert np.allclose(quantiles, [expected[k] for k in ("q10", "q50", "q90")], atol=1e-6, rtol=0)
print({"completed": len(powers), "power_kW_quantiles": quantiles.tolist()})


{'completed': 200, 'power_kW_quantiles': [312.6605070606375, 335.3363172109298, 359.0628331245354]}


In [5]:
generate_chapter("ch10")
print("Chapter illustrations regenerated from maintained source and verified results.")

Chapter illustrations regenerated from maintained source and verified results.


![Compressor sensitivity](../figures/compressor_sensitivity.png)

Read the corresponding chapter discussion for the diagram's meaning or the numerical figure's observation, mechanism, implication and recommendation.

![Compressor uncertainty](../figures/compressor_uncertainty.png)

Read the corresponding chapter discussion for the diagram's meaning or the numerical figure's observation, mechanism, implication and recommendation.

![Gas process](../figures/gas_process.png)

Read the corresponding chapter discussion for the diagram's meaning or the numerical figure's observation, mechanism, implication and recommendation.

![Process workspace](../figures/process_workspace.png)

Read the corresponding chapter discussion for the diagram's meaning or the numerical figure's observation, mechanism, implication and recommendation.